# 21장 · 검색된 문단으로 답할 수 없을 때

B형의 파일을 무선으로 옮길 수 있는지 묻자 검색기는 파일 복사 페이지를 찾습니다. 그쪽에는 케이블 사용법만 있습니다. “관련 있는 쪽을 찾았다”에서 “무선 전송을 지원한다”로 뛰어넘는 순간, 학생에게 틀린 사용법을 알려 주게 됩니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 질문을 주장으로 나누어 보기

“B형 녹음 파일을 컴퓨터에 무선으로 보낼 수 있나요?”에서 기종·대상·전송 방식을 표시합니다. 소스 PDF의 2쪽을 읽어 **케이블 복사**, **무선 전송 지원**, **무선 전송 미지원** 세 주장 중 어떤 것을 원문만으로 확인할 수 있는지 판단하세요. 지원 방식 목록이 모두 실린 문서인지 모르므로, 언급이 없다는 이유로 미지원이라고 단정하기도 어렵습니다.

In [2]:
from rag_lab import open_index, retrieve, answer_from_documents
index = open_index()
q = "2027년 5월 B형에서 녹음 파일을 무선으로 컴퓨터에 보낼 수 있나요?"
candidates = retrieve(index, q, k=3)
for row in candidates:
    print(row["id"], row["text"][100:235])

recorder-b-1.0.pdf#p2 이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한
recorder-a-1.1.pdf#p2 이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한
recorder-a-1.0.pdf#p2 립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다.


검색 거리가 짧다는 것은 주제가 가깝다는 뜻입니다. 쪽에 ‘컴퓨터’, ‘파일’, ‘복사’가 함께 나와도 무선 지원을 확인할 수는 없습니다. 답변 지시에는 사실을 뒷받침하는 범위와, 근거가 부족할 때 사용자에게 알려 줄 정보를 함께 적습니다.

In [3]:
print(answer_from_documents(q, candidates))
# 같은 방식으로 근거가 분명한 질문도 확인합니다.
supported = "2027년 5월 B형에서 케이블로 연결한 뒤 어느 폴더를 열까요?"
print(answer_from_documents(supported, retrieve(index, supported, k=3)))

2027년 5월 B형에서 녹음 파일을 무선으로 컴퓨터에 보낼 수 없습니다. 자료에 따르면, B형 1.0판(2027년 4월 1일부터 적용)에서는 녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하여 녹음기에 있는 RECORD 폴더에서 파일을 복사하도록 안내하고 있습니다. 무선 전송에 대한 언급은 없습니다.[recorder-b-1.0.pdf#p2]
2027년 5월 B형에서 케이블로 연결한 뒤에는 녹음기에 있는 RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다[recorder-b-1.0.pdf#p2].


## 무조건 보류하는 답도 실패입니다

지원 여부가 불분명하면 확인을 보류하고, 문서가 다루는 범위가 케이블 연결임을 안내할 수 있습니다. 반대로 두 번째 질문에는 RECORD 폴더라는 근거가 있으므로 답해야 합니다. “확인할 수 없습니다”만 모든 질문에 내놓는 도우미는 첫 사례만 우연히 맞힙니다.

## 답의 표현과 증거의 표현 구별하기

검색에 실패했을 때 가장 위험한 오답 하나를 원문과 나란히 놓으세요. 아래 문장은 오류를 찾도록 만든 가상 답입니다. 판정할 때는 제품명이나 인용 표시가 그럴듯한지가 아니라, ‘무선 전송을 지원하지 않는다’는 주장이 실제 쪽에 있는지를 읽습니다.

In [4]:
from rag_lab import SOURCES, read_pages
manual_page = read_pages(SOURCES / "recorder-b-1.0.pdf")[1]
print("검토할 가상 답: B형은 무선 전송을 지원하지 않습니다.")
print(manual_page["text"][manual_page["text"].index("파일 옮기기"):])

검토할 가상 답: B형은 무선 전송을 지원하지 않습니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한 파일은
남습니다. 파일을 컴퓨터에 복사한 뒤 녹음기에서 지워 공간을
확보하세요.
가상 제품 설명서   B형 1.0판
2 / 2쪽


케이블 복사가 된다는 안내는 보입니다. 전송 방식을 빠짐없이 나열한다고 선언한 문장도, 무선 기능이 없다는 문장도 없습니다. 따라서 “무선 지원 여부는 확인하기 어렵고, 문서로 확인되는 케이블 복사는 이렇게 한다”로 수정합니다. 프로그램으로 단어 하나의 존재만 검사해서 이 판단을 대신하지 않습니다.

사용자에게는 보류 이유와 다음 행동이 필요합니다. 제품이 무엇인지 몰라 답할 수 없을 때는 기종을 묻고, 기종은 알지만 설명서에 근거가 없을 때는 확인한 문서와 그 범위를 알려 주세요. 케이블 복사라는 대안을 안내할 때도 그것이 무선 전송 절차인 것처럼 표현하지 않습니다.

## 실습 · 추가 질문과 보류를 구분하기

① “녹음 중 표시등을 숨길 수 있나요?” ② “2027년 5월 B형의 무선 전송 규격은 무엇인가요?” ③ “같은 B형으로 녹음 중에 표시등 설정을 바꿀 수 있나요?”를 직접 보냅니다. ①에는 추가로 알아야 할 조건, ②에는 문서에 빠진 사실, ③에는 답을 뒷받침할 쪽을 적습니다.

In [5]:
# 질문별로 후보와 답을 모두 저장하고 PDF를 엽니다.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>for question in [&quot;녹음 중 표시등을 숨길 수 있나요?&quot;,
                 &quot;2027년 5월 B형의 무선 전송 규격은 무엇인가요?&quot;,
                 &quot;2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 있나요?&quot;]:
    docs = retrieve(index, question, k=4)
    print(question, answer_from_documents(question, docs))</code></pre><p>①은 기종을 물어보거나 기종별 차이를 조건부로 안내할 수 있습니다. ②는 규격을 찾지 못했다고 밝혀야 합니다. ③은 B형 1.0판 1쪽에 따라 녹음 전 설정해야 한다고 안내합니다. 세 번째까지 보류했다면 관련 쪽이 후보에 있었는지부터 보세요.</p><p>집필 시 실행 결과</p><pre>녹음 중 표시등을 숨길 수 있나요? 들소리 A형(1.0판, 1.1판)에서는 녹음 중 표시등을 끄는 설정이 없습니다. 녹음 중에는 앞면의 빨간 표시등이 켜집니다. 화면 밝기를 낮춰도 표시등은 켜져 있습니다. [recorder-a-1.0.pdf#p1], [recorder-a-1.1.pdf#p1]

들소리 B형(1.0판)에서는 녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸면 녹음을 시작해도 앞면에 빨간불이 들어오지 않습니다. 단, 이미 녹음 중이면 표시등 설정을 바꿀 수 없고, 녹음을 끝낸 뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작해야 합니다. [recorder-b-1.0.pdf#p1]

따라서, 녹음 중 표시등을 숨길 수 있는 기종은 B형 1.0판이며, A형 1.0판과 1.1판에서는 불가능합니다.
2027년 5월 B형의 무선 전송 규격은 무엇인가요? 확인 보류: 2027년 5월 B형의 무선 전송 규격에 관한 정보가 자료에 없습니다.
2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 있나요? 2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 없습니다.  
자료에 따르면, B형 1.0판(적용 기간 2027년 4월 1일부터)에서 &quot;이미 녹음 중이면 표시등 설정을 바꿀 수 없습니다. 녹음을 끝낸 뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작하세요.&quot;라고 명시되어 있습니다.[recorder-b-1.0.pdf#p1]</pre></details>

## 독립 실습 · 보류를 잘 쓴 답 고르기

다음 문구는 **오류를 찾도록 만든 세 가지 가상 답변**입니다. B형 2쪽을 읽고 사실로 확인되는 구절에만 밑줄을 그으세요. 그런 다음 문의자가 지금 할 수 있는 행동을 담아 다시 씁니다.

- “무선 전송을 지원하지 않습니다.”
- “케이블로 파일을 복사할 수 있습니다. 무선 전송 지원 여부는 주어진 설명서에서 확인되지 않습니다.”
- “블루투스를 켜고 기기 이름을 선택하세요.”

다른 학생은 같은 설명서로 “컴퓨터에 어떤 폴더가 보이나요?”를 묻습니다. 보류 문구를 그대로 반복해도 되는지 실행해 확인하세요.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>케이블로 파일을 복사할 수 있다는 두 번째 답이 자료 범위에 맞습니다. 설명서가 특정 방법 한 가지를 안내했다고 해서 모든 다른 방법을 금지한다고 단정할 수 없습니다. 폴더 이름은 B형 1.0판 2쪽에서 RECORD라고 확인할 수 있으므로 보류하지 않고 직접 답합니다.</p></details>

## 다음 수업에 가져갈 것

근거가 있는 질문과 없는 질문을 짝으로 남기세요. 다음 장에서 정답 쪽이 검색되는 문제와, 검색돼도 답할 수 없는 문제를 따로 평가합니다.